In [1]:
# Cryptogram Project
# Author: Althea Doherty
# Description: creating a cryptogram puzzle

In [33]:
#importing libraries needed for GUI
import tkinter as tk

#creating a pop out window for program
window = tk.Tk()
window.geometry("850x450")  #window size
window.title("Cryptogram Puzzle")  #window title

# This function runs every time the user types in an Entry box.
# newVal = the value the Entry *will have* after the keystroke.
def validChar(newVal):
    if newVal == "":
        return True  # allow deleting the character
    return len(newVal) == 1 and newVal.isalpha()  # allow only ONE letter

# Tkinter requires registering the validation function
vcmd = (window.register(validChar), "%P")

health = tk.IntVar(value=100)  #health bar status variable
hLabel = tk.Label(window, text="Health: 100")  #health bar label text
hLabel.pack(pady=10)  #spacing for health label

#label on top of where quote will sit
eLabel = tk.Label(window, text="Quote")
eLabel.pack(pady=10)

#label on top of where entry window is
pLabel = tk.Label(window, text="Progress")
pLabel.pack(pady=10)

#frame to hold encrypted letters + entry boxes
pFrame = tk.Frame(window)
pFrame.pack(pady=5)

#frame to hold guess history
gFrame = tk.Frame(window)
gFrame.pack(pady=5)

encrypted ="Qrwklqj'v shupdqhqw."  #encrypted text shown to player
correct = "NOTHING'S PERMANENT."  #correct decrypted solution

# Extract only alphabetic characters from correct answer
# Used later for autofill logic
corAns = [ch.upper() for ch in correct if ch.isalpha()]

#list to hold entry widgets for each letter
entries = []

for i, ch in enumerate(encrypted):
    # Display encrypted character above entry box
    label = tk.Label(pFrame, text=ch, font=("Times",12))
    label.grid(row=0, column=i, padx=3)

    if ch.isalpha():  #only create entry box if character is a letter
        entry = tk.Entry(pFrame,width=3,font=("Terminal",12),justify="center",validate="key", validatecommand=vcmd)
        entry.grid(row=1, column=i, padx=3)
        entries.append(entry)  #store entry for later checking
    else:
        #placeholder for non-letters (spaces, punctuation)
        tk.Label(pFrame, text=" ").grid(row=1, column=i)

def hLvl(amount):
    #reduce health but never below 0
    hupdate = max(0, health.get() - amount)
    health.set(hupdate)
    hLabel.config(text=f"Health: {hupdate}")  #update label text

def subGuess():
    #get all guesses from entry boxes, convert to uppercase
    guessl = (e.get().upper() for e in entries)
    wrong = False  #track if any guess is wrong

    # FIRST PASS: check each guess and color boxes
    for i, guess in enumerate(guessl):
        if guess == correct[i]:  #correct letter
            entries[i].config(bg="OliveDrab1")
        else:  #incorrect letter
            entries[i].config(bg="tomato")
            wrong = True

    #if any wrong guesses, reduce health
    if wrong:
        hLvl(10)

    # SECOND PASS: auto-fill matching letters in all positions
    guessl = (e.get().upper() for e in entries)  #regenerate generator
    for i, guess in enumerate(guessl):
        # ensure guess exists and index is valid
        if guess and i < len(corAns) and guess == corAns[i]:
            # fill all matching positions with the same letter
            for x, actual in enumerate(corAns):
                if actual == guess:
                    entries[i].delete(0, tk.END)
                    entries[i].insert(0, guess)
                    entries[i].config(bg="OliveDrab1")

    row = tk.Frame(gFrame)
    row.pack()

    # Display each guess with its color (green/red)
    for i, e in enumerate(entries):
        tk.Label(row,text=e.get().upper(),width=3,font=("Terminal",14),  justify="center",bg=e.cget("bg")).grid(row=0, column=i, padx=3, pady=3)

def restartGame():
    # reset health
    health.set(100)
    hLabel.config(text="Health: 100")

    # clear all entry boxes and reset colors
    for e in entries:
        e.delete(0, tk.END)
        e.config(bg="white")

    # clear guess history frame
    for widget in gFrame.winfo_children():
        widget.destroy()

subButton = tk.Button(window, text="Submit Entry", command=subGuess)
subButton.pack(pady=10)

restartButton = tk.Button(window, text="Restart Game", command=restartGame)
restartButton.pack(pady=5)

window.mainloop()